# RL-дообучение GPT-2 рекомендера на KION

План: данные и разбиение по времени → базовая GPT-2 → reward model → RL (REINFORCE/GRPO + KL) → анализ.
Статус: каркас, секции заполняются по мере работы.

In [ ]:
import os, glob, json, random
from pathlib import Path
import numpy as np

ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = not ON_KAGGLE   # локально: малая выборка; на Kaggle: полный прогон (для быстрой проверки на Kaggle поставьте True)
SEED = 42

WORK = Path("/kaggle/working") if ON_KAGGLE else Path(".")
CACHE, CKPT, RESULTS = WORK/"cache", WORK/"checkpoints", WORK/"results"
for d in (CACHE, CKPT, RESULTS): d.mkdir(exist_ok=True)

def find_data_dir():
    if os.environ.get("DATA_DIR"): return Path(os.environ["DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/interactions.csv", recursive=True)
    if not hits: raise FileNotFoundError("interactions.csv не найден: подключите датасет или задайте DATA_DIR")
    return Path(hits[0]).parent
DATA_DIR = find_data_dir()

random.seed(SEED); np.random.seed(SEED)
print("kaggle:", ON_KAGGLE, "| smoke:", SMOKE, "| data:", DATA_DIR)

## 1. Данные и разбиение по времени

**Протокол разбиения** (один для всех моделей):

- Глобальные даты: train `< 2021-08-02`, val `2021-08-02 … 08-08`, test `2021-08-09 … 08-22` (последние 14 дней).
- Обучение (GPT-2, reward model, RL) использует только train. Val нужна для подбора настроек, test открываем один раз в конце.
- Eval-кейс: пользователь с ≥5 событиями в train и ≥1 событием в периоде. Цель — первое событие в периоде, контекст — все события пользователя до начала периода (для test это train+val). Следующие 5 событий периода сохраняем как «будущее» для проверки 5-шаговой генерации.
- Объекты, которых нет в train, из val/test выкидываем (холодный старт здесь не исследуем), долю считаем и печатаем.
- В KION есть только дата без времени: порядок внутри дня берём из порядка строк файла (стабильная сортировка). Это шум, он описан в ограничениях.
- Пара (пользователь, объект) в данных не повторяется, поэтому уже просмотренные объекты исключаем из рекомендаций при оценке.

In [ ]:
import pandas as pd

TRAIN_END = pd.Timestamp("2021-08-02")   # train: dt < TRAIN_END
VAL_END   = pd.Timestamp("2021-08-09")   # val: [TRAIN_END, VAL_END), test: >= VAL_END
MIN_HIST  = 5                            # минимум событий в train
N_FUTURE  = 5                            # длина «будущего» для многошагового анализа
SMOKE_USERS = 30_000                     # размер выборки пользователей при SMOKE=True
TAG = "smoke" if SMOKE else "full"

In [ ]:
def build_split():
    df = pd.read_csv(DATA_DIR/"interactions.csv", parse_dates=["last_watch_dt"])
    n_raw = len(df)
    df = df.dropna(subset=["watched_pct"]).reset_index(drop=True)
    df["row"] = np.arange(len(df))
    df = df.sort_values(["user_id", "last_watch_dt", "row"], kind="mergesort").reset_index(drop=True)
    df["period"] = np.select([df.last_watch_dt < TRAIN_END, df.last_watch_dt < VAL_END], [0, 1], 2)  # 0 train, 1 val, 2 test

    n_train = df[df.period == 0].groupby("user_id").size()
    keep = n_train[n_train >= MIN_HIST].index
    if SMOKE:
        keep = pd.Index(np.random.RandomState(SEED).choice(keep, SMOKE_USERS, replace=False))
    df = df[df.user_id.isin(keep)].copy()

    # словарь объектов только по train; 0 зарезервирован под PAD
    train_items = np.sort(df.loc[df.period == 0, "item_id"].unique())
    item2idx = {it: i + 1 for i, it in enumerate(train_items)}
    seen = df.item_id.isin(item2idx)
    unseen_share = {p: float((~seen[df.period == p]).mean()) for p in (1, 2)}
    df = df[seen].copy()
    df["item"] = df.item_id.map(item2idx).astype("int32")
    df = df[["user_id", "item", "item_id", "last_watch_dt", "watched_pct", "total_dur", "period"]].reset_index(drop=True)
    return df, item2idx, unseen_share, n_raw

cache_f = CACHE/f"interactions_{TAG}.parquet"
if cache_f.exists():
    df = pd.read_parquet(cache_f)
    item2idx = {it: i + 1 for i, it in enumerate(np.sort(df.loc[df.period == 0, "item_id"].unique()))}
    unseen_share = json.load(open(RESULTS/f"split_stats_{TAG}.json"))["unseen_share"]
else:
    df, item2idx, unseen_share, n_raw = build_split()
    df.to_parquet(cache_f)
n_items = len(item2idx) + 1   # с учётом PAD=0
print(df.shape, "users:", df.user_id.nunique(), "| vocab:", n_items)

Eval-кейсы: цель, контекст (индексы строк в `df`) и «будущее» из 5 событий.

In [ ]:
def build_eval(df, period):
    """Одна строка на пользователя: первое событие периода — цель, следующие N_FUTURE — будущее."""
    cur = df[df.period == period]
    g = cur.groupby("user_id", sort=False)
    first = g.head(1).set_index("user_id")
    fut = g.head(1 + N_FUTURE).groupby("user_id", sort=False).item.apply(lambda s: s.iloc[1:].tolist())
    out = pd.DataFrame({"target": first["item"], "target_pct": first["watched_pct"],
                        "target_dt": first["last_watch_dt"], "future": fut})
    return out

evals = {"val": build_eval(df, 1), "test": build_eval(df, 2)}

def history(user_id, period):
    """Контекст пользователя: все события строго до начала периода."""
    u = df_by_user.get_group(user_id)
    return u[u.period < period]

df_by_user = df.groupby("user_id", sort=False)
for k, e in evals.items(): print(k, "eval users:", len(e))

Проверки на утечки и статистика.

In [ ]:
tr, va, te = (df[df.period == p] for p in (0, 1, 2))
assert tr.last_watch_dt.max() < TRAIN_END
assert va.last_watch_dt.between(TRAIN_END, VAL_END - pd.Timedelta(days=1)).all()
assert te.last_watch_dt.min() >= VAL_END
assert df.groupby("user_id").last_watch_dt.apply(lambda s: s.is_monotonic_increasing).all()   # сортировка по времени
assert set(va.item) <= set(tr.item) and set(te.item) <= set(tr.item)                             # словарь только из train
assert not df.duplicated(["user_id", "item"]).any()                                              # нет повторов, значит цель не из истории
for k, p in (("val", 1), ("test", 2)):                                                           # цель всегда позже истории
    e = evals[k]; hist_last = df[df.period < p].groupby("user_id").last_watch_dt.max()
    assert (e.target_dt.values >= hist_last.reindex(e.index).values).all()

stats = {
    "tag": TAG, "users": int(df.user_id.nunique()), "vocab": int(n_items),
    "events": {"train": len(tr), "val": len(va), "test": len(te)},
    "train_seq_len_median": float(tr.groupby("user_id").size().median()),
    "eval_users": {k: len(e) for k, e in evals.items()},
    "unseen_share": unseen_share,
    "watched_pct_mean": {"train": float(tr.watched_pct.mean()), "val": float(va.watched_pct.mean()), "test": float(te.watched_pct.mean())},
    "share_pct_ge_50": {"train": float((tr.watched_pct >= 50).mean()), "test": float((te.watched_pct >= 50).mean())},
}
json.dump(stats, open(RESULTS/f"split_stats_{TAG}.json", "w"), indent=1)
print(json.dumps(stats, indent=1)); print("утечек не найдено")

## 2. Базовая модель (GPT-2, next item prediction)

**Модель и протокол оценки.**

- GPT-2 (HF `GPT2LMHeadModel`, веса с нуля), токен = индекс объекта, 0 = PAD. Контекст — последние 50 объектов.
- Обучение: train-период, окна по 50 объектов со сдвигом 25, loss = кросс-энтропия next item.
- Оценка: ранжируем весь каталог, **исключая уже просмотренные объекты и PAD**. Единственный релевантный объект — цель кейса, поэтому Recall@K = HitRate@K, а MAP@K = MRR@K.
- Все модели (бейзлайны, база, позже RL) оцениваются одной функцией `evaluate`. Ничья по score считается как худший ранг.
- Доверительные интервалы 95%: бутстреп по пользователям (1000 повторов); для разности моделей — парный бутстреп.
- Разнообразие: покрытие каталога в топ-10, Gini экспозиции объектов, средняя лог-популярность рекомендаций.

In [ ]:
import torch, time
from transformers import GPT2Config, GPT2LMHeadModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN, KS, K_MAX = 50, (5, 10, 20), 20
CFG = (dict(n_layer=2, n_head=4, n_embd=128, epochs=3, bs=64, lr=1e-3) if SMOKE
       else dict(n_layer=4, n_head=4, n_embd=256, epochs=6, bs=256, lr=5e-4))
torch.manual_seed(SEED)
print(DEVICE, CFG)

item_cnt = np.bincount(df.loc[df.period == 0, "item"].values, minlength=n_items)   # популярность в train

def pad(seqs, dtype=np.int64):
    a = np.zeros((len(seqs), max(map(len, seqs))), dtype)
    for i, s in enumerate(seqs): a[i, :len(s)] = s
    return torch.from_numpy(a)

_hist_cache = {}
def histories(split):
    """Полная история каждого eval-пользователя: все события строго до начала периода."""
    if split in _hist_cache: return _hist_cache[split]
    p = {"val": 1, "test": 2}[split]
    sub = df[df.period < p]; uid, it = sub.user_id.values, sub.item.values
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    pos = pd.Series(np.arange(len(starts)), index=uid[starts]).reindex(evals[split].index).values
    _hist_cache[split] = [it[starts[j]:ends[j]] for j in pos]
    return _hist_cache[split]

@torch.no_grad()
def evaluate(score_fn, split, bs=512, limit=None):
    """score_fn(ctx[B,L], lens[B]) -> scores[B,V]. Возвращает ранг цели и топ-K_MAX рекомендаций."""
    e, hist = evals[split].iloc[:limit], histories(split)[:limit]
    tgt = e.target.values.astype(np.int64)
    ranks, recs = np.empty(len(e), np.int64), np.empty((len(e), K_MAX), np.int64)
    for i in range(0, len(e), bs):
        hb = hist[i:i + bs]
        ctx = pad([h[-MAX_LEN:] for h in hb]); lens = torch.tensor([min(len(h), MAX_LEN) for h in hb])
        sc = score_fn(ctx.to(DEVICE), lens.to(DEVICE)).clone()
        sc[:, 0] = -float("inf")
        rows = torch.from_numpy(np.concatenate([np.full(len(h), j) for j, h in enumerate(hb)]))
        cols = torch.from_numpy(np.concatenate(hb).astype(np.int64))
        sc[rows.to(DEVICE), cols.to(DEVICE)] = -float("inf")           # просмотренное не рекомендуем
        t = torch.from_numpy(tgt[i:i + bs]).to(DEVICE)
        ranks[i:i + bs] = (sc >= sc.gather(1, t[:, None])).sum(1).cpu().numpy()
        recs[i:i + bs] = sc.topk(K_MAX, 1).indices.cpu().numpy()
    return ranks, recs

def user_metrics(ranks):
    m = {}
    for k in KS:
        hit = ranks <= k
        m[f"Recall@{k}"], m[f"NDCG@{k}"], m[f"MAP@{k}"] = hit * 1.0, hit / np.log2(ranks + 1), hit / ranks
    return m

def boot_ci(x, B=1000, seed=0):
    """Среднее и 95% ДИ бутстрепом по пользователям."""
    rng, n, means = np.random.default_rng(seed), len(x), np.empty(B)
    for s in range(0, B, 100):
        means[s:s + 100] = x[rng.integers(0, n, (min(100, B - s), n))].mean(1)
    return [float(x.mean()), *map(float, np.percentile(means, [2.5, 97.5]))]

def diversity(recs, k=10):
    r = recs[:, :k].ravel(); cnt = np.bincount(r, minlength=n_items)[1:]
    p = np.sort(cnt) / cnt.sum(); n = len(p)
    return {"coverage@10": float((cnt > 0).mean()), "gini@10": float((2 * np.arange(1, n + 1) - n - 1) @ p / n),
            "mean_log_pop@10": float(np.log1p(item_cnt[r]).mean())}

def diversity_ci(recs, B=200, seed=0):
    rng, base = np.random.default_rng(seed), diversity(recs)
    boots = [diversity(recs[rng.integers(0, len(recs), len(recs))]) for _ in range(B)]
    q = {k: np.percentile([b[k] for b in boots], [2.5, 97.5]) for k in base}
    out = {k: [v, *map(float, q[k])] for k, v in base.items()}
    out["coverage@10"] = [base["coverage@10"], None, None]   # при пересэмплировании пользователей покрытие смещается вниз: ДИ не считаем
    return out

def report(ranks, recs):
    return {**{k: boot_ci(v) for k, v in user_metrics(ranks).items()}, **diversity_ci(recs)}

def table(res, cols=("Recall@10", "NDCG@10", "MAP@10", "Recall@20", "coverage@10", "gini@10", "mean_log_pop@10")):
    fmt = lambda x: f"{x[0]:.4f}" if x[1] is None else f"{x[0]:.4f} [{x[1]:.4f}, {x[2]:.4f}]"
    return pd.DataFrame({m: {c: fmt(r[c]) for c in cols} for m, r in res.items()}).T

### Бейзлайны: популярное за всё время и за последние 7 дней train

In [ ]:
last7 = df[(df.period == 0) & (df.last_watch_dt >= TRAIN_END - pd.Timedelta(days=7))]
cnt7 = np.bincount(last7.item.values, minlength=n_items)
pop_all = torch.tensor(item_cnt, dtype=torch.float32, device=DEVICE)
pop_7d = torch.tensor(cnt7 + 1e-6 * item_cnt, dtype=torch.float32, device=DEVICE)   # хвост разруливаем общей популярностью

val_res, val_raw = {}, {}
for name, vec in (("Pop (all time)", pop_all), ("Pop (last 7d)", pop_7d)):
    r, rec = evaluate(lambda c, l, v=vec: v[None].expand(len(c), -1), "val")
    val_raw[name] = (r, rec); val_res[name] = report(r, rec)
table(val_res)

### Обучение GPT-2

In [ ]:
def make_windows(stride=MAX_LEN // 2):
    tr = df[df.period == 0]; uid, it = tr.user_id.values, tr.item.values
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    out = []
    for s, e in zip(starts, ends):
        seq = it[s:e]
        if len(seq) <= MAX_LEN: out.append(seq); continue
        out += [seq[end - MAX_LEN:end] for end in range(len(seq), MAX_LEN - 1, -stride)]
        if (len(seq) - MAX_LEN) % stride: out.append(seq[:MAX_LEN])
    return out

def new_model():
    cfg = GPT2Config(vocab_size=n_items, n_positions=MAX_LEN, n_embd=CFG["n_embd"], n_layer=CFG["n_layer"],
                     n_head=CFG["n_head"], pad_token_id=0, bos_token_id=0, eos_token_id=0)
    return GPT2LMHeadModel(cfg).to(DEVICE)

def gpt_scores(model):
    def f(ctx, lens):
        h = model.transformer(input_ids=ctx, attention_mask=(ctx != 0).long()).last_hidden_state
        return model.lm_head(h[torch.arange(len(ctx), device=ctx.device), lens - 1]).float()
    return f

def train_base(model, seqs, log_path):
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    steps = CFG["epochs"] * ((len(seqs) + CFG["bs"] - 1) // CFG["bs"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, CFG["lr"], total_steps=steps, pct_start=0.1)
    amp = DEVICE == "cuda"; scaler = torch.amp.GradScaler(enabled=amp)
    log, best = [], -1
    for ep in range(CFG["epochs"]):
        model.train(); t0, losses = time.time(), []
        perm = np.random.permutation(len(seqs))
        for i in range(0, len(seqs), CFG["bs"]):
            x = pad([seqs[j] for j in perm[i:i + CFG["bs"]]]).to(DEVICE)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=amp):
                loss = model(input_ids=x, attention_mask=(x != 0).long(), labels=x.masked_fill(x == 0, -100)).loss
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        model.eval(); r, _ = evaluate(gpt_scores(model), "val"); rec10 = float((r <= 10).mean())
        log.append(dict(epoch=ep + 1, train_loss=float(np.mean(losses)), val_recall10=rec10, sec=round(time.time() - t0)))
        print(log[-1])
        if rec10 > best: best = rec10; torch.save(model.state_dict(), CKPT/f"base_{TAG}.pt")
    pd.DataFrame(log).to_csv(log_path, index=False)

seqs = make_windows(); print("train windows:", len(seqs))
base = new_model(); print("params:", sum(p.numel() for p in base.parameters()))
if (CKPT/f"base_{TAG}.pt").exists():
    print("загружаю чекпойнт")
else:
    train_base(base, seqs, RESULTS/f"base_train_log_{TAG}.csv")
base.load_state_dict(torch.load(CKPT/f"base_{TAG}.pt", map_location=DEVICE)); _ = base.eval()

### Базовая модель на val против бейзлайнов

In [ ]:
r, rec = evaluate(gpt_scores(base), "val")
val_raw["GPT-2 base"] = (r, rec); val_res["GPT-2 base"] = report(r, rec)
json.dump(val_res, open(RESULTS/f"base_val_{TAG}.json", "w"), indent=1)
display(table(val_res))

# парная разность с лучшим бейзлайном
best_bl = max(("Pop (all time)", "Pop (last 7d)"), key=lambda n: val_res[n]["NDCG@10"][0])
for m in ("Recall@10", "NDCG@10"):
    d = user_metrics(val_raw["GPT-2 base"][0])[m] - user_metrics(val_raw[best_bl][0])[m]
    c = boot_ci(d)
    print(f"{m}: GPT-2 base - {best_bl} = {c[0]:.4f}, 95% ДИ [{c[1]:.4f}, {c[2]:.4f}]")

### Выводы по базовой модели

_Заполнить числами после полного прогона на Kaggle._

## 3. Reward model (watched_pct)

**Постановка.** Reward model (RM) предсказывает `watched_pct / 100` для пары (история пользователя, объект-кандидат).

- Архитектура: каузальный GPT-2-энкодер над историей даёт вектор `h_t`; предсказание для события `t+1` = `sigmoid(MLP([h_t, emb(item_{t+1})]))`. За один проход по окну из 50 событий получаем предсказания для всех позиций.
- Вход истории: эмбеддинг объекта + эмбеддинг его `watched_pct` (уровень вовлечённости пользователя — самый сильный сигнал). Во время обучения у 50% окон отклики заменяются на обучаемый вектор «неизвестно», чтобы RM работала и на сгенерированных последовательностях, где откликов нет.
- Обучение: train-период, MSE. Оценка на val: все события val (контекст — всё, что было раньше у пользователя, включая более ранние события val; берём последние 50 событий).
- Сравнение с бейзлайнами без контекста: константа, среднее по объекту, среднее по пользователю, их сумма (все со сглаживанием по 5 событиям, считаются по train).
- Метрики: MAE, RMSE, AUC для «досмотрел» (`watched_pct ≥ 50`), Spearman; ДИ бутстрепом по пользователям (200 повторов); калибровка по децилям.
- RM обучена только на реальных историях. На сгенерированных последовательностях (секция 6) она работает вне распределения, это учтём в выводах.

In [ ]:
from torch import nn
from transformers import GPT2Model
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

RM_CFG = (dict(n_layer=2, n_head=4, n_embd=128, epochs=2, bs=32, lr=1e-3) if SMOKE
          else dict(n_layer=2, n_head=4, n_embd=256, epochs=3, bs=256, lr=5e-4))

class RewardModel(nn.Module):
    def __init__(self, n_items, d, n_layer, n_head):
        super().__init__()
        self.backbone = GPT2Model(GPT2Config(vocab_size=n_items, n_positions=MAX_LEN, n_embd=d, n_layer=n_layer, n_head=n_head))
        self.cand = nn.Embedding(n_items, d, padding_idx=0)      # отдельные эмбеддинги кандидата
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, 1))
        self.pct_proj = nn.Sequential(nn.Linear(1, d), nn.GELU(), nn.Linear(d, d))   # отклик на прошлое событие
        self.unk = nn.Parameter(torch.zeros(d))                                      # «отклик неизвестен»
    def hidden(self, ctx, pct=None):
        """pct[B,L] в [0,1] или -1 (неизвестно); pct=None: все отклики неизвестны."""
        e = self.backbone.wte(ctx)
        if pct is None: pct = torch.full(ctx.shape, -1.0, device=ctx.device)
        pe = torch.where((pct >= 0)[..., None], self.pct_proj(pct.clamp(min=0)[..., None].to(e.dtype)), self.unk.to(e.dtype))
        return self.backbone(inputs_embeds=e + pe, attention_mask=(ctx != 0).long()).last_hidden_state
    def score(self, h, cand):                                    # h[...,d], cand[...] -> доля просмотра в (0,1)
        return torch.sigmoid(self.head(torch.cat([h, self.cand(cand)], -1)).squeeze(-1))

@torch.no_grad()
def rm_candidates(rm, ctx, lens, cand, ctx_pct=None):
    """Предсказанная доля просмотра для кандидатов cand[B,K] при контексте ctx[B,L] (понадобится в RL)."""
    h = rm.hidden(ctx, ctx_pct)[torch.arange(len(ctx), device=ctx.device), lens - 1]
    return rm.score(h[:, None, :].expand(-1, cand.shape[1], -1), cand)

def window_ranges(n, stride=MAX_LEN // 2):
    if n <= MAX_LEN: return [(0, n)]
    r = [(e - MAX_LEN, e) for e in range(n, MAX_LEN - 1, -stride)]
    return r + ([(0, MAX_LEN)] if (n - MAX_LEN) % stride else [])

def rm_train_windows():
    tr = df[df.period == 0]; uid, it, y = tr.user_id.values, tr.item.values, tr.watched_pct.values / 100
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    return [(it[s + a:s + b], y[s + a:s + b].astype(np.float32)) for s, e in zip(starts, ends) for a, b in window_ranges(e - s)]

_rmw_cache = {}
def rm_windows(split):
    """Последние 50 событий каждого eval-пользователя до конца периода; flag — событие принадлежит периоду."""
    if split in _rmw_cache: return _rmw_cache[split]
    p = {"val": 1, "test": 2}[split]
    sub = df[df.period <= p]; uid, it, y, per = sub.user_id.values, sub.item.values, sub.watched_pct.values / 100, sub.period.values
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    pos = pd.Series(np.arange(len(starts)), index=uid[starts]).reindex(evals[split].index).values
    sl = [(max(starts[j], ends[j] - MAX_LEN), ends[j]) for j in pos]
    _rmw_cache[split] = ([it[a:b] for a, b in sl], [y[a:b].astype(np.float32) for a, b in sl], [per[a:b] == p for a, b in sl])
    return _rmw_cache[split]

@torch.no_grad()
def rm_predict(rm, split, use_pct=True, bs=512):
    """Предсказания (в процентах) для всех событий периода, у которых есть хотя бы одно событие контекста.
    use_pct: подавать ли реальные отклики контекста (False: «неизвестно»)."""
    items, ys, flags = rm_windows(split); rm.eval()
    P, Y, U, I = [], [], [], []
    for i in range(0, len(items), bs):
        X = pad(items[i:i + bs]).to(DEVICE); Yb = pad(ys[i:i + bs], np.float32); Fb = pad(flags[i:i + bs])[:, 1:].bool()
        pred = rm.score(rm.hidden(X, Yb.to(DEVICE) if use_pct else None)[:, :-1], X[:, 1:]).float().cpu()
        m = Fb & (X[:, 1:].cpu() != 0)
        u = torch.arange(i, i + len(X))[:, None].expand_as(m)
        P.append(pred[m]); Y.append(Yb[:, 1:][m]); U.append(u[m]); I.append(X[:, 1:].cpu()[m])
    cat = lambda l: torch.cat(l).numpy()
    return cat(P) * 100, cat(Y) * 100, cat(U), cat(I)

def train_rm(rm, wins, log_path):
    c = RM_CFG
    opt = torch.optim.AdamW(rm.parameters(), lr=c["lr"], weight_decay=0.01)
    steps = c["epochs"] * ((len(wins) + c["bs"] - 1) // c["bs"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, c["lr"], total_steps=steps, pct_start=0.1)
    amp = DEVICE == "cuda"; scaler = torch.amp.GradScaler(enabled=amp)
    log, best = [], 1e9
    for ep in range(c["epochs"]):
        rm.train(); t0, losses = time.time(), []
        perm = np.random.permutation(len(wins))
        for i in range(0, len(wins), c["bs"]):
            b = [wins[j] for j in perm[i:i + c["bs"]]]
            X = pad([w[0] for w in b]).to(DEVICE); Y = pad([w[1] for w in b], np.float32).to(DEVICE)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=amp):
                known = torch.rand(len(X), 1, device=DEVICE) < 0.5          # у половины окон отклики скрыты
                pred = rm.score(rm.hidden(X, torch.where(known, Y, torch.full_like(Y, -1.0)))[:, :-1], X[:, 1:])
            v = X[:, 1:] != 0
            loss = ((pred.float() - Y[:, 1:]) ** 2)[v].mean()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        p, y, _, _ = rm_predict(rm, "val"); rmse = float(np.sqrt(((p - y) ** 2).mean()))
        log.append(dict(epoch=ep + 1, train_mse=float(np.mean(losses)) * 1e4, val_rmse=rmse, sec=round(time.time() - t0)))
        print(log[-1])
        if rmse < best: best = rmse; torch.save(rm.state_dict(), CKPT/f"rm_{TAG}.pt")
    pd.DataFrame(log).to_csv(log_path, index=False)

rm = RewardModel(n_items, RM_CFG["n_embd"], RM_CFG["n_layer"], RM_CFG["n_head"]).to(DEVICE)
print("RM params:", sum(p.numel() for p in rm.parameters()))
if not (CKPT/f"rm_{TAG}.pt").exists():
    train_rm(rm, rm_train_windows(), RESULTS/f"rm_train_log_{TAG}.csv")
rm.load_state_dict(torch.load(CKPT/f"rm_{TAG}.pt", map_location=DEVICE)); _ = rm.eval()

### Оценка reward model на val

In [ ]:
p_rm, y, uidx, it_val = rm_predict(rm, "val")
n_users_val = len(evals["val"])
tr = df[df.period == 0]; K_SH = 5; g_mean = tr.watched_pct.mean()
im_sum = np.bincount(tr.item.values, weights=tr.watched_pct.values, minlength=n_items)
item_mean = (im_sum + K_SH * g_mean) / (item_cnt + K_SH)                           # среднее по объекту, сглаженное
u_g = tr.groupby("user_id").watched_pct.agg(["sum", "count"])
user_mean = ((u_g["sum"] + K_SH * g_mean) / (u_g["count"] + K_SH)).reindex(evals["val"].index[uidx]).values
preds = {"Константа": np.full_like(y, g_mean), "Среднее по объекту": item_mean[it_val],
         "Среднее по пользователю": user_mean,
         "Объект + пользователь": np.clip(item_mean[it_val] + user_mean - g_mean, 0, 100),
         "RM (отклики контекста неизвестны)": rm_predict(rm, "val", use_pct=False)[0], "RM (отклики известны)": p_rm}

def boot_reg(preds, y, uidx, n_users, B=200, seed=0):
    rng, lab = np.random.default_rng(seed), y >= 50
    n_u = np.bincount(uidx, minlength=n_users).astype(float)
    s_abs = {m: np.bincount(uidx, weights=np.abs(p - y), minlength=n_users) for m, p in preds.items()}
    s_sq = {m: np.bincount(uidx, weights=(p - y) ** 2, minlength=n_users) for m, p in preds.items()}
    pt = {m: dict(MAE=float(np.abs(p - y).mean()), RMSE=float(np.sqrt(((p - y) ** 2).mean())), AUC=float(roc_auc_score(lab, p)),
                  Spearman=float(spearmanr(p, y)[0])) for m, p in preds.items()}
    bs = {m: dict(MAE=[], RMSE=[], AUC=[]) for m in preds}; d_rmse = {m: [] for m in preds}
    for _ in range(B):
        w = np.bincount(rng.integers(0, n_users, n_users), minlength=n_users).astype(float); den = w @ n_u
        for m, p in preds.items():
            bs[m]["MAE"].append(w @ s_abs[m] / den); bs[m]["RMSE"].append(np.sqrt(w @ s_sq[m] / den))
            bs[m]["AUC"].append(roc_auc_score(lab, p, sample_weight=w[uidx]))
        for m in preds: d_rmse[m].append(bs[m]["RMSE"][-1] - bs["Среднее по объекту"]["RMSE"][-1])
    ci = lambda a: [float(x) for x in np.percentile(a, [2.5, 97.5])]
    res = {m: {**pt[m], **{f"{k}_ci": ci(v) for k, v in bs[m].items()}, "dRMSE_vs_item_mean_ci": ci(d_rmse[m])} for m in preds}
    return res

rm_res = boot_reg(preds, y, uidx, n_users_val)
json.dump(rm_res, open(RESULTS/f"rm_val_{TAG}.json", "w"), indent=1)
fmt = lambda r, k: f"{r[k]:.4f} [{r[k + '_ci'][0]:.4f}, {r[k + '_ci'][1]:.4f}]"
print("событий val:", len(y), "| доля watched>=50:", round(float((y >= 50).mean()), 3))
pd.DataFrame({m: {**{k: fmt(r, k) for k in ("MAE", "RMSE", "AUC")}, "Spearman": f"{r['Spearman']:.4f}",
                  "ΔRMSE к 'среднее по объекту'": f"[{r['dRMSE_vs_item_mean_ci'][0]:.3f}, {r['dRMSE_vs_item_mean_ci'][1]:.3f}]"}
              for m, r in rm_res.items()}).T

Калибровка по децилям предсказания и проверка согласованности `rm_candidates` с обучающим проходом.

In [ ]:
q = pd.qcut(p_rm, 10, labels=False, duplicates="drop")
calib = pd.DataFrame({"pred": [p_rm[q == b].mean() for b in range(q.max() + 1)], "true": [y[q == b].mean() for b in range(q.max() + 1)]}).round(2)
calib.to_csv(RESULTS/f"rm_calibration_{TAG}.csv", index=False); print(calib.T.to_string())

# prefix-инвариантность: предсказание последнего события окна == rm_candidates по укороченному контексту
items_v, ys_v, _ = rm_windows("val"); keep_v = [i for i, w in enumerate(items_v) if len(w) >= 4][:64]
ws = [items_v[i] for i in keep_v]; ys_v = [ys_v[i] for i in keep_v]
with torch.no_grad():
    X = pad(ws).to(DEVICE); L = torch.tensor([len(w) for w in ws], device=DEVICE); ar = torch.arange(len(ws), device=DEVICE)
    Yw = pad(ys_v[:len(ws)], np.float32).to(DEVICE)
    full = rm.score(rm.hidden(X, Yw)[ar, L - 2], X[ar, L - 1])
    ctx = pad([w[:-1] for w in ws]).to(DEVICE); cp = pad([w[:-1] for w in ys_v[:len(ws)]], np.float32).to(DEVICE)
    part = rm_candidates(rm, ctx, L - 1, X[ar, L - 1][:, None], cp)[:, 0]
print("max |diff|:", float((full - part).abs().max()))
assert float((full - part).abs().max()) < 1e-4

### Выводы по reward model

_Заполнить числами после полного прогона на Kaggle._

## 4. RL-дообучение (REINFORCE/GRPO + KL)

**Метод.** Политика π_θ — копия базовой GPT-2, референс π_ref — она же, заморожённая. На каждом шаге:

1. Берём батч реальных контекстов из train (префикс окна), для каждого сэмплируем G траекторий длины H из π_θ (просмотренное маскируется).
2. Вознаграждение траектории — сумма (с дисконтом γ) предсказаний reward model для каждого сгенерированного объекта. Контекст RM: реальная история с известными откликами + ранее сгенерированные объекты с откликом «неизвестен».
3. GRPO: преимущество = награда − среднее по группе, нормировка на std по батчу. Loss = −A·Σ log π(a_t|s_t) + β·KL(π_θ‖π_ref), KL считается точно по всему каталогу на посещённых состояниях.

**Функции вознаграждения** (`spec`):

| имя | награда |
|---|---|
| `pct` | RM-A: предсказанная доля просмотра |
| `pct+hit` | `pct` + 0.5, если объект совпал с реальным следующим (логированный сигнал, прямой якорь на релевантность) |
| `pct+div` | `pct` − η·ln(c), c — сколько раз объект выбран в батче у разных пользователей (штраф за «всем одно и то же»); η = 0.01, в `pct+div_hi` η = 0.05 |
| `long` | RM-ret: вероятность вернуться в течение 3 дней |
| `pct+long` | сумма `pct` и `long` |

**Контроль reward hacking:** независимая RM-B (другая глубина и seed, в RL не участвует). Если reward по RM-A растёт, а по RM-B нет, модель эксплуатирует ошибки RM-A.

In [ ]:
# справочники по объектам: длительность (оценка), тип контента
items_meta = pd.read_csv(DATA_DIR/"items.csv", usecols=["item_id", "content_type"]).drop_duplicates("item_id").set_index("item_id")
idx2id = np.zeros(n_items, np.int64)
for it_, i_ in item2idx.items(): idx2id[i_] = it_
item_series = np.r_[False, items_meta.content_type.reindex(idx2id[1:]).values == "series"]

trn = df[df.period == 0]; ok_ = (trn.watched_pct >= 20) & (trn.total_dur > 0)
dur = (trn.total_dur[ok_] / (trn.watched_pct[ok_] / 100)).groupby(trn.item[ok_]).median() / 60   # минуты: total_dur / доля просмотра
item_dur = np.full(n_items, dur.median(), np.float32); item_dur[dur.index.values] = dur.values
print(f"доля сериалов в каталоге: {item_series[1:].mean():.2f} | медиана оценки длительности: {np.median(item_dur[1:]):.0f} мин")

# долгосрочная цель: пользователь вернулся (новое событие в другой день) в течение H_RET дней
H_RET = 3
day = lambda ts: int(np.datetime64(ts).astype("datetime64[D]").astype(np.int64))
dd = df.last_watch_dt.values.astype("datetime64[D]").astype(np.int64)
d_ = pd.DataFrame({"user_id": df.user_id.values, "d": dd})
ud = d_.drop_duplicates().copy(); ud["nxt"] = ud.groupby("user_id").d.shift(-1)
nxt = d_.merge(ud, on=["user_id", "d"], how="left").nxt.values
df["ret"] = ((nxt - dd) <= H_RET).astype(np.float32)
end_day = np.select([df.period.values == 0, df.period.values == 1], [day(TRAIN_END) - 1, day(VAL_END) - 1], dd.max())
df["ret_valid"] = (dd + H_RET <= end_day)                 # горизонт целиком внутри периода (нет цензурирования)
df["pct01"], df["one"] = (df.watched_pct / 100).astype(np.float32), np.float32(1)
print("доля возвратов за 3 дня (train, valid):", round(float(df[(df.period == 0) & df.ret_valid].ret.mean()), 3),
      "| val:", round(float(df[(df.period == 1) & df.ret_valid].ret.mean()), 3))

### 4.1 Дополнительные reward model: долгосрочная (RM-ret) и независимый судья (RM-B)

In [ ]:
import torch.nn.functional as F

def long_windows(tcol, vcol):
    t = df[df.period == 0]; uid, it = t.user_id.values, t.item.values
    pc, tg, vl = (t[c].values.astype(np.float32) for c in ("pct01", tcol, vcol))
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    return [(it[s + a:s + b], pc[s + a:s + b], tg[s + a:s + b], vl[s + a:s + b]) for s, e in zip(starts, ends) for a, b in window_ranges(e - s)]

def fit_rm(rm_, wins, kind, ckpt, val_fn, cfg, log_path):
    """kind='pct': MSE по доле просмотра; kind='ret': BCE по бинарной цели. val_fn(rm) -> число, меньше = лучше."""
    opt = torch.optim.AdamW(rm_.parameters(), lr=cfg["lr"], weight_decay=0.01)
    steps = cfg["epochs"] * ((len(wins) + cfg["bs"] - 1) // cfg["bs"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, cfg["lr"], total_steps=steps, pct_start=0.1)
    amp = DEVICE == "cuda"; scaler = torch.amp.GradScaler(enabled=amp); log, best = [], 1e9
    for ep in range(cfg["epochs"]):
        rm_.train(); t0, losses = time.time(), []
        perm = np.random.permutation(len(wins))
        for i in range(0, len(wins), cfg["bs"]):
            b = [wins[j] for j in perm[i:i + cfg["bs"]]]
            X = pad([w[0] for w in b]).to(DEVICE)
            PC, T, V = (pad([w[k] for w in b], np.float32).to(DEVICE) for k in (1, 2, 3))
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=amp):
                known = torch.rand(len(X), 1, device=DEVICE) < 0.5
                pred = rm_.score(rm_.hidden(X, torch.where(known, PC, torch.full_like(PC, -1.0)))[:, :-1], X[:, 1:])
            m = (X[:, 1:] != 0) & (V[:, 1:] > 0)
            if not m.any(): continue
            pr, tg = pred.float()[m], T[:, 1:][m]
            loss = ((pr - tg) ** 2).mean() if kind == "pct" else F.binary_cross_entropy(pr.clamp(1e-6, 1 - 1e-6), tg)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(rm_.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        v = val_fn(rm_); log.append(dict(epoch=ep + 1, train_loss=float(np.mean(losses)), val_metric=v, sec=round(time.time() - t0))); print(log[-1])
        if v < best: best = v; torch.save(rm_.state_dict(), ckpt)
    pd.DataFrame(log).to_csv(log_path, index=False)

def ret_val(rm_):
    """Предсказания RM-ret на событиях val с полным горизонтом."""
    p, _, u, i = rm_predict(rm_, "val"); p = p / 100
    key = evals["val"].index.values[u].astype(np.int64) * 100000 + i
    v = df[df.period == 1]; vk = v.user_id.values.astype(np.int64) * 100000 + v.item.values
    pos = pd.Series(np.arange(len(vk)), index=vk).reindex(key).values.astype(np.int64)
    ok = v.ret_valid.values[pos]
    return p[ok], v.ret.values[pos][ok], u[ok], i[ok]

# RM-B: независимый судья по доле просмотра (1 слой, другой seed)
def val_rmse(m):
    p, y_, _, _ = rm_predict(m, "val")
    return float(np.sqrt(((p - y_) ** 2).mean()))

RM_B_CFG = dict(RM_CFG, n_layer=1)
torch.manual_seed(SEED + 1)
rm_b = RewardModel(n_items, RM_B_CFG["n_embd"], RM_B_CFG["n_layer"], RM_B_CFG["n_head"]).to(DEVICE)
if not (CKPT/f"rm_b_{TAG}.pt").exists():
    fit_rm(rm_b, long_windows("pct01", "one"), "pct", CKPT/f"rm_b_{TAG}.pt",
           val_rmse, RM_B_CFG, RESULTS/f"rm_b_train_log_{TAG}.csv")
rm_b.load_state_dict(torch.load(CKPT/f"rm_b_{TAG}.pt", map_location=DEVICE)); _ = rm_b.eval()

# RM-ret: вероятность вернуться в течение 3 дней
def ret_auc_loss(m):
    p, y_, _, _ = ret_val(m)
    return 1 - float(roc_auc_score(y_, p))

rm_ret = RewardModel(n_items, RM_CFG["n_embd"], RM_CFG["n_layer"], RM_CFG["n_head"]).to(DEVICE)
if not (CKPT/f"rm_ret_{TAG}.pt").exists():
    fit_rm(rm_ret, long_windows("ret", "ret_valid"), "ret", CKPT/f"rm_ret_{TAG}.pt",
           ret_auc_loss, RM_CFG, RESULTS/f"rm_ret_train_log_{TAG}.csv")
rm_ret.load_state_dict(torch.load(CKPT/f"rm_ret_{TAG}.pt", map_location=DEVICE)); _ = rm_ret.eval()
judges = {"RM-A": rm, "RM-B": rm_b, "RM-ret": rm_ret}

In [ ]:
# качество RM-B (по доле просмотра) и RM-ret (возврат за 3 дня) на val
pb, yb, ub, ib = rm_predict(rm_b, "val")
print("RM-B  RMSE:", round(float(np.sqrt(((pb - yb) ** 2).mean())), 3), "| AUC watched>=50:", round(float(roc_auc_score(yb >= 50, pb)), 4),
      "| corr(RM-A, RM-B):", round(float(np.corrcoef(p_rm, pb)[0, 1]), 3))

def auc_ci(p, y, u, n_users, B=100, seed=0):
    rng, out = np.random.default_rng(seed), []
    for _ in range(B):
        w = np.bincount(rng.integers(0, n_users, n_users), minlength=n_users).astype(float)
        out.append(roc_auc_score(y, p, sample_weight=w[u]))
    return [float(roc_auc_score(y, p)), *map(float, np.percentile(out, [2.5, 97.5]))]

pr_, yr_, ur_, ir_ = ret_val(rm_ret)
tv = df[(df.period == 0) & df.ret_valid]; g_ret = tv.ret.mean()
ir_rate = (np.bincount(tv.item.values, weights=tv.ret.values, minlength=n_items) + 5 * g_ret) / (np.bincount(tv.item.values, minlength=n_items) + 5)
ug = tv.groupby("user_id").ret.agg(["sum", "count"])
ur_rate = ((ug["sum"] + 5 * g_ret) / (ug["count"] + 5)).reindex(evals["val"].index[ur_]).fillna(g_ret).values
ret_res = {"Среднее по объекту": ir_rate[ir_], "Среднее по пользователю": ur_rate, "RM-ret": pr_}
ret_tab = {k: auc_ci(v, yr_, ur_, len(evals["val"])) for k, v in ret_res.items()}
json.dump(ret_tab, open(RESULTS/f"rm_ret_val_{TAG}.json", "w"), indent=1)
print("RM-ret, событий val с полным горизонтом:", len(yr_), "| доля возвратов:", round(float(yr_.mean()), 3))
pd.DataFrame({k: {"AUC": f"{v[0]:.4f} [{v[1]:.4f}, {v[2]:.4f}]"} for k, v in ret_tab.items()}).T

### 4.2 Инфраструктура RL: контексты, награды, метрики поведения, GRPO

In [ ]:
_ctx_cache = {}
def contexts(split):
    """Реальные контексты eval-пользователей: последние 50 событий до периода, их отклики известны."""
    if split in _ctx_cache: return _ctx_cache[split]
    p = {"val": 1, "test": 2}[split]
    sub = df[df.period < p]; uid, pc = sub.user_id.values, sub.pct01.values
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    pos = pd.Series(np.arange(len(starts)), index=uid[starts]).reindex(evals[split].index).values
    hp = [pc[starts[j]:ends[j]][-MAX_LEN:] for j in pos]
    hist = [h[-MAX_LEN:] for h in histories(split)]
    _ctx_cache[split] = (pad(hist), torch.tensor([len(h) for h in hist]), pad(hp, np.float32))
    return _ctx_cache[split]

@torch.no_grad()
def reward_at_k(recs, split, k=10, bs=512, limit=None):
    """Средний предсказанный reward топ-k рекомендаций на пользователя по каждой RM (контекст реальный)."""
    ctx, clen, cp = (t[:limit] for t in contexts(split)); out = {n: [] for n in judges}
    for i in range(0, len(ctx), bs):
        c, l, q = ctx[i:i + bs].to(DEVICE), clen[i:i + bs].to(DEVICE), cp[i:i + bs].to(DEVICE)
        cand = torch.from_numpy(recs[i:i + bs, :k]).to(DEVICE)
        for n, m in judges.items(): out[n].append(rm_candidates(m, c, l, cand, q).mean(1).cpu())
    return {n: torch.cat(v).numpy() for n, v in out.items()}

def behavior(recs, base_recs=None, k=10):
    r = recs[:, :k]
    out = {**diversity(recs, k), "mean_dur_min": float(item_dur[r].mean()), "share_series": float(item_series[r].mean()),
           "mean_item_pct_prior": float(item_mean[r].mean())}
    if base_recs is not None:
        out["overlap_with_base@10"] = float((r[:, :, None] == base_recs[:, None, :k]).any(-1).mean())
    return out

BASE_RECS = None
def eval_recs(name, ranks, recs, split, base_recs=None):
    """Метрики релевантности, reward@10 по всем RM и поведение для готовых рекомендаций."""
    rw = reward_at_k(recs, split)
    np.save(CACHE/f"recs_{name}_{split}_{RUN_TAG}.npy", recs); np.save(CACHE/f"ranks_{name}_{split}_{RUN_TAG}.npy", ranks)
    return {"rank_metrics": report(ranks, recs), "reward@10": {n: boot_ci(v) for n, v in rw.items()},
            "behavior": behavior(recs, base_recs)}

def full_eval(name, model, split="val", base_recs=None):
    ranks, recs = evaluate(gpt_scores(model), split)
    return eval_recs(name, ranks, recs, split, BASE_RECS if base_recs is None else base_recs), recs

def append_tok(ctx, clen, cpct, a):
    full = clen >= MAX_LEN
    ctx = torch.where(full[:, None], torch.roll(ctx, -1, 1), ctx); cpct = torch.where(full[:, None], torch.roll(cpct, -1, 1), cpct)
    pos = clen.clamp(max=MAX_LEN - 1)[:, None]
    return (ctx.scatter(1, pos, a[:, None]), cpct.scatter(1, pos, torch.full((len(a), 1), -1.0, device=a.device)),
            (clen + 1).clamp(max=MAX_LEN))

def masked_logits(model, ctx, clen):
    lg = gpt_scores(model)(ctx, clen)
    mask = torch.zeros_like(lg, dtype=torch.bool); mask[:, 0] = True; mask.scatter_(1, ctx, True)    # PAD и уже виденное
    return lg.masked_fill(mask, -1e4)

def step_reward(spec, a, ctx, clen, cpct, target):
    with torch.no_grad():
        r = torch.zeros(len(a), device=a.device)
        if spec.get("w_pct"): r += spec["w_pct"] * rm_candidates(rm, ctx, clen, a[:, None], cpct)[:, 0]
        if spec.get("w_ret"): r += spec["w_ret"] * rm_candidates(rm_ret, ctx, clen, a[:, None], cpct)[:, 0]
        if spec.get("hit") and target is not None: r += spec["hit"] * (a == target).float()
        if spec.get("div"): r -= spec["div"] * torch.log(torch.bincount(a, minlength=n_items)[a].float())
    return r

RL_CFG = (dict(B=32, G=4, steps=40, lr=1e-4, log_every=10) if SMOKE else dict(B=128, G=8, steps=250, lr=3e-5, log_every=25))
RUN_TAG = f"{TAG}_s{RL_CFG['steps']}_lr{RL_CFG['lr']:g}"      # имена файлов RL зависят от настроек: старые результаты не подхватятся
rl_wins = rm_train_windows()          # (items, доля просмотра) окна train

def sample_contexts(B, rng):
    ctxs, cps, tg = [], [], []
    for j in rng.integers(0, len(rl_wins), B):
        it, pc = rl_wins[j]; t = rng.integers(3, len(it))
        ctxs.append(it[:t]); cps.append(pc[:t]); tg.append(it[t])
    full = lambda x: F.pad(x, (0, MAX_LEN - x.shape[1]))            # ширина MAX_LEN нужна для append_tok
    return (full(pad(ctxs)).to(DEVICE), torch.tensor([len(c) for c in ctxs]).to(DEVICE),
            full(pad(cps, np.float32)).to(DEVICE), torch.tensor(tg, dtype=torch.long).to(DEVICE))

def quick_eval(model, n=2000):
    r, rec = evaluate(gpt_scores(model), "val", limit=n)
    return float((r <= 10).mean()), float(reward_at_k(rec, "val", limit=n)["RM-A"].mean())

def rl_train(name, reward, beta, H=1, gamma=0.9, seed=0, **over):
    c = {**RL_CFG, **over}; B, G = c["B"], c["G"]
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    policy = new_model(); policy.load_state_dict(base.state_dict()); policy.eval()
    opt = torch.optim.AdamW(policy.parameters(), lr=c["lr"], weight_decay=0.0)
    log = []
    for step in range(c["steps"] + 1):
        if step % c["log_every"] == 0:
            rec10, rw = quick_eval(policy); log.append(dict(step=step, val_recall10_sub=rec10, val_rmA_top10=rw))
        if step == c["steps"]: break
        ctx, clen, cpct, target = (x.repeat_interleave(G, 0) for x in sample_contexts(B, rng))
        R = torch.zeros(len(ctx), device=DEVICE); logp_sum = torch.zeros_like(R); kl_sum = torch.zeros_like(R); ent0 = hit0 = None
        for t in range(H):
            lg = masked_logits(policy, ctx, clen)
            with torch.no_grad(): lq = F.log_softmax(masked_logits(base, ctx, clen), -1)
            lp = F.log_softmax(lg, -1); p = lp.exp()
            a = torch.multinomial(p.detach(), 1)[:, 0]
            logp_sum += lp.gather(1, a[:, None])[:, 0]; kl_sum += (p * (lp - lq)).sum(1)
            R += gamma ** t * step_reward(reward, a, ctx, clen, cpct, target if t == 0 else None)
            if t == 0: ent0, hit0 = float(-(p * lp).sum(1).mean().detach()), float((a == target).float().mean())
            ctx, cpct, clen = append_tok(ctx, clen, cpct, a)
        A = R.view(B, G) - R.view(B, G).mean(1, keepdim=True); A = (A / (A.std() + 1e-6)).view(-1)
        loss = -(A.detach() * logp_sum).mean() + beta * kl_sum.mean()
        opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step()
        log[-1].update(reward=float(R.mean()), kl=float(kl_sum.mean().detach()), entropy=ent0, sample_hit_rate=hit0)
        if step % c["log_every"] == 0: print(name, {k: round(v, 4) for k, v in log[-1].items()})
    pd.DataFrame(log).to_csv(RESULTS/f"rl_log_{name}_{RUN_TAG}.csv", index=False)
    torch.save(policy.state_dict(), CKPT/f"rl_{name}_{RUN_TAG}.pt")
    return policy

Референс: базовая модель на тех же метриках, затем пробный прогон `pct`, β=0.1.

In [ ]:
base_out, BASE_RECS = full_eval("base", base)
json.dump(base_out, open(RESULTS/f"rl_base_{RUN_TAG}.json", "w"), indent=1)
print("base reward@10:", {k: round(v[0], 4) for k, v in base_out["reward@10"].items()})
print("base behavior:", {k: round(v, 3) for k, v in base_out["behavior"].items()})
pilot = rl_train("pilot", dict(w_pct=1), beta=0.5, steps=RL_CFG["steps"] // 4)
del pilot
if DEVICE == "cuda": torch.cuda.empty_cache()

### 4.3 Сетка прогонов

β-сетка для `pct` (0.1…8) даёт кривую «reward — релевантность»; остальные награды запускаются с β=0.5, `long_b0.1` показывает, что делает слабый KL. `pct_H5` — горизонт 5 шагов (меньший батч, при нехватке памяти повтор с вдвое меньшим).

In [ ]:
BETAS = (0.1, 0.5, 2.0, 8.0)
RL_RUNS = {f"pct_b{b:g}": dict(reward=dict(w_pct=1), beta=b) for b in BETAS}
RL_RUNS.update({
    "pct+hit": dict(reward=dict(w_pct=1, hit=0.5), beta=0.5),
    "pct+div": dict(reward=dict(w_pct=1, div=0.01), beta=0.5),
    "pct+div_hi": dict(reward=dict(w_pct=1, div=0.05), beta=0.5),
    "long": dict(reward=dict(w_ret=1), beta=0.5),
    "long_b0.1": dict(reward=dict(w_ret=1), beta=0.1),
    "pct+long": dict(reward=dict(w_pct=1, w_ret=1), beta=0.5),
    "pct_H5": dict(reward=dict(w_pct=1), beta=0.5, H=5, B=RL_CFG["B"] // 4),
})
rl_results, rl_errors = {"base": base_out}, {}
for name, s_ in RL_RUNS.items():
    f = RESULTS/f"rl_{name}_{RUN_TAG}.json"
    if f.exists(): rl_results[name] = json.load(open(f)); continue
    spec = dict(s_)
    for attempt in range(2):                              # при нехватке памяти повторяем с вдвое меньшим батчем
        try:
            pol = rl_train(name, **spec)
            rl_results[name], _ = full_eval(name, pol); json.dump(rl_results[name], open(f, "w"), indent=1)
            del pol; break
        except torch.cuda.OutOfMemoryError:
            pol = None
            if DEVICE == "cuda": torch.cuda.empty_cache()
            spec["B"] = spec.get("B", RL_CFG["B"]) // 2; print(name, "OOM, повтор с B =", spec["B"])
        except Exception as ex:                           # один сбой не срывает остальные прогоны
            rl_errors[name] = repr(ex); print(name, "ОШИБКА:", ex); break
    if DEVICE == "cuda": torch.cuda.empty_cache()
print("не завершились:", rl_errors or "нет")

def rl_table(res):
    g = lambda r, k: f"{r['rank_metrics'][k][0]:.4f}"
    rows = {n: {"Recall@10": g(r, "Recall@10"), "NDCG@10": g(r, "NDCG@10"), "rwd RM-A": f"{r['reward@10']['RM-A'][0]:.3f}",
                "rwd RM-B": f"{r['reward@10']['RM-B'][0]:.3f}", "rwd ret": f"{r['reward@10']['RM-ret'][0]:.3f}",
                "overlap w/ base": f"{r['behavior'].get('overlap_with_base@10', 1):.2f}", "coverage": f"{r['behavior']['coverage@10']:.3f}",
                "gini": f"{r['behavior']['gini@10']:.3f}", "dur, мин": f"{r['behavior']['mean_dur_min']:.0f}",
                "сериалы": f"{r['behavior']['share_series']:.2f}"} for n, r in res.items()}
    return pd.DataFrame(rows).T
rl_table(rl_results)

### Выводы по RL

_Заполнить после полного прогона на Kaggle._

## 5. Анализ: reward vs релевантность, разнообразие, reward hacking

Все сравнения на val, одинаковый протокол. Рекомендации и ранги каждого прогона сохранены в `cache/` (`recs_*`, `ranks_*`).

- **5.1** Кривая «reward — релевантность» и траектории обучения.
- **5.2** Разности с базой с ДИ (парный бутстреп по пользователям).
- **5.3** Как меняется поведение: популярность, длительность, сериалы, покрытие, схлопывание политики.
- **5.4** Кто теряет качество: по длине истории и уровню вовлечённости.
- **5.5** Какие объекты получают экспозицию.
- **5.6** Проверка без RM: фактический досмотр и возврат этих объектов в val.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "font.size": 10})
FAM = {"pct_b": ("#0072B2", "o"), "pct+hit": ("#009E73", "s"), "pct+div": ("#E69F00", "^"), "long": ("#D55E00", "v"),
       "pct+long": ("#CC79A7", "D"), "pct_H5": ("#56B4E9", "P")}
def fam(name):
    for k in ("pct+long", "pct+hit", "pct+div", "pct_H5", "pct_b", "long"):
        if name.startswith(k): return FAM[k]
    return ("#444444", "o")

runs = [n for n in rl_results if n != "base"]
ld = lambda kind, n, split="val": np.load(CACHE/f"{kind}_{n}_{split}_{RUN_TAG}.npy")
logs = {n: pd.read_csv(RESULTS/f"rl_log_{n}_{RUN_TAG}.csv") for n in runs if (RESULTS/f"rl_log_{n}_{RUN_TAG}.csv").exists()}

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for n, r in rl_results.items():
    x = r["reward@10"]["RM-A"][0]; y, lo, hi = r["rank_metrics"]["Recall@10"]; col, mk = ("#222222", "*") if n == "base" else fam(n)
    ax[0].errorbar(x, y, yerr=[[y - lo], [hi - y]], fmt=mk, color=col, ms=9 if n == "base" else 7, capsize=2)
    ax[0].annotate(n, (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8)
ax[0].set(xlabel="средний reward RM-A в топ-10", ylabel="Recall@10", title="Итог: reward против релевантности (val)")
for n, lg in logs.items():
    col, mk = fam(n); ax[1].plot(lg.val_rmA_top10, lg.val_recall10_sub, color=col, marker=mk, ms=3, lw=1, alpha=.8)
    ax[1].annotate(n, (lg.val_rmA_top10.iloc[-1], lg.val_recall10_sub.iloc[-1]), textcoords="offset points", xytext=(4, 2), fontsize=8)
ax[1].set(xlabel="средний reward RM-A в топ-10 (подвыборка 2000 пользователей)", ylabel="Recall@10 (подвыборка)", title="Траектории обучения (от базы к концу)")
plt.tight_layout(); plt.savefig(RESULTS/f"fig_pareto_{RUN_TAG}.png", dpi=130); plt.show()

### 5.2 Разности с базой (парный бутстреп по пользователям, 95% ДИ)

In [ ]:
base_um, base_recs_v = user_metrics(ld("ranks", "base")), ld("recs", "base"); base_rw = reward_at_k(base_recs_v, "val")
delta = {}
for n in runs:
    um, rw = user_metrics(ld("ranks", n)), reward_at_k(ld("recs", n), "val")
    delta[n] = {"ΔRecall@10": boot_ci(um["Recall@10"] - base_um["Recall@10"]), "ΔNDCG@10": boot_ci(um["NDCG@10"] - base_um["NDCG@10"]),
                "ΔRM-A": boot_ci(rw["RM-A"] - base_rw["RM-A"]), "ΔRM-B": boot_ci(rw["RM-B"] - base_rw["RM-B"]),
                "ΔRM-ret": boot_ci(rw["RM-ret"] - base_rw["RM-ret"])}
json.dump(delta, open(RESULTS/f"rl_delta_{RUN_TAG}.json", "w"), indent=1)
fmt_d = lambda v: f"{v[0]:+.4f} [{v[1]:+.4f}, {v[2]:+.4f}]"
pd.DataFrame({n: {k: fmt_d(v) for k, v in d.items()} for n, d in delta.items()}).T

### 5.3 Поведение: популярность, длительность, сериалы, схлопывание политики

In [ ]:
def last_stats(n):
    lg = logs.get(n); return {} if lg is None else lg.dropna(subset=["kl"]).iloc[-1][["kl", "entropy"]].to_dict()
beh = {n: {**r["behavior"], **last_stats(n)} for n, r in rl_results.items()}
cols = ["mean_log_pop@10", "mean_dur_min", "share_series", "mean_item_pct_prior", "coverage@10", "gini@10", "overlap_with_base@10", "kl", "entropy"]
pd.DataFrame(beh).T.reindex(columns=cols).round(3)

### 5.4 Кто теряет качество: длина истории и уровень вовлечённости

In [ ]:
hl = np.array([len(h) for h in histories("val")])
ctx_v, clen_v, cp_v = contexts("val")
eng = ((cp_v * (ctx_v != 0)).sum(1) / clen_v.clamp(min=1)).numpy()                       # средний досмотр в истории
groups = {"история 5–9": hl < 10, "история 10–24": (hl >= 10) & (hl < 25), "история ≥25": hl >= 25}
q1, q2 = np.quantile(eng, [1 / 3, 2 / 3])
groups.update({"вовлечённость: низкая": eng <= q1, "вовлечённость: средняя": (eng > q1) & (eng <= q2), "вовлечённость: высокая": eng > q2})
sel = [n for n in ("pct_b0.5", "pct+hit", "pct+div", "long_b0.1", "pct_H5") if n in rl_results]
rk = {n: ld("ranks", n) for n in ["base"] + sel}
seg = {g: {"n": int(m.sum()), **{n: float((rk[n][m] <= 10).mean()) for n in rk}} for g, m in groups.items()}
seg = pd.DataFrame(seg).T
for n in sel: seg[f"Δ {n}"] = seg[n] - seg["base"]
seg.round(4)

### 5.5 Какие объекты получают экспозицию

In [ ]:
meta = pd.read_csv(DATA_DIR/"items.csv", usecols=["item_id", "title", "content_type"]).drop_duplicates("item_id").set_index("item_id")
expo = lambda recs: np.bincount(recs[:, :10].ravel(), minlength=n_items) / recs[:, :10].size
e0 = expo(base_recs_v)
print("доля слотов у топ-100 объектов: base =", round(float(np.sort(e0)[::-1][:100].sum()), 3))
for n in [x for x in ("pct_b0.5", "long_b0.1") if x in rl_results]:
    e1 = expo(ld("recs", n)); d = e1 - e0; top = np.argsort(-d)[:8]
    print(f"\n{n}: доля слотов у топ-100 объектов = {np.sort(e1)[::-1][:100].sum():.3f}; объекты с наибольшим ростом экспозиции")
    display(pd.DataFrame({"название": meta.title.reindex(idx2id[top]).values, "тип": meta.content_type.reindex(idx2id[top]).values,
                          "доля слотов было": e0[top].round(4), "стало": e1[top].round(4), "длит., мин": item_dur[top].round(),
                          "ср. досмотр в train, %": item_mean[top].round(1), "просмотров в train": item_cnt[top]}))

### 5.6 Проверка без reward model: что реально случилось с этими объектами в val

Для рекомендованных объектов считаем **фактический** средний `watched_pct` и долю возвратов за 3 дня по событиям val (минимум 20 событий на объект). Эти числа RM не использовали, поэтому они показывают, не эксплуатирует ли политика ошибки RM. Оговорка: это средние по объекту, а не персональные исходы; объекты без поддержки в таблицу не попадают (доля с поддержкой указана).

In [ ]:
v_ = df[df.period == 1]
ia = v_.groupby("item").watched_pct.agg(["size", "mean"]); ir = v_[v_.ret_valid].groupby("item").ret.agg(["size", "mean"])
obs_pct = np.full(n_items, np.nan); obs_pct[ia.index[ia["size"] >= 20]] = ia["mean"][ia["size"] >= 20]
obs_ret = np.full(n_items, np.nan); obs_ret[ir.index[ir["size"] >= 20]] = ir["mean"][ir["size"] >= 20]
def obs(recs):
    r = recs[:, :10]
    return {"факт. досмотр, %": float(np.nanmean(obs_pct[r])), "факт. возврат 3д": float(np.nanmean(obs_ret[r])),
            "доля слотов с поддержкой": float(np.isfinite(obs_pct[r]).mean())}
obs_tab = pd.DataFrame({n: obs(ld("recs", n)) for n in ["base"] + runs}).T
obs_tab["ΔRM-A к базе"] = [0.0] + [delta[n]["ΔRM-A"][0] for n in runs]
obs_tab.to_csv(RESULTS/f"rl_observed_{RUN_TAG}.csv"); obs_tab.round(4)

### Выводы по разделу 5

_Заполнить после полного прогона на Kaggle._

## 6. Генерация на 5 шагов вперёд

**Постановка.** Для пользователей val, у которых есть минимум 5 реальных событий в периоде (цель + 4 следующих), строим последовательность из 5 шагов: на каждом шаге берём топ-1, добавляем его в контекст, рекомендуем дальше. Два режима:

- **free** — в контекст добавляется собственная рекомендация (то, что произойдёт на практике при цепочке рекомендаций);
- **teacher** — в контекст добавляется реальное событие пользователя (контроль: что было бы, если бы пользователь следовал реальной траектории).

Метрики по шагам t = 1…5: Recall@10 относительно реального t-го события, reward RM-A и RM-B топ-1 (у добавленных объектов отклик «неизвестен»), доля уникальных объектов среди пользователей на шаге t (схлопывание), доля реальных событий, попавших в сгенерированную цепочку (sequence recall). Разность free − teacher показывает цену того, что сгенерированные объекты не являются настоящими взаимодействиями. Уже предложенные объекты маскируются, поэтому в free-режиме объект, угаданный раньше, на следующем шаге недоступен: это занижает Recall free, а sequence recall от этого свободен.

In [ ]:
H_MS, N_MS = 5, 20000
lpop_t = torch.log1p(torch.tensor(item_cnt, dtype=torch.float32, device=DEVICE))

@torch.no_grad()
def multistep(model, H=H_MS, n_max=N_MS, bs=512):
    e = evals["val"]; elig = np.array([len(f) + 1 >= H for f in e.future.values])
    idx = np.sort(np.random.default_rng(0).permutation(np.flatnonzero(elig))[:n_max])
    ctx_all, clen_all, cp_all = contexts("val")
    real = torch.tensor(np.stack([[e.target.values[i], *e.future.values[i][:H - 1]] for i in idx]), dtype=torch.long)
    res = {"n_users": int(len(idx))}
    for mode in ("free", "teacher"):
        per = {k: [[] for _ in range(H)] for k in ("hit", "rmA", "rmB", "lpop")}; gen = []
        for i in range(0, len(idx), bs):
            sl = idx[i:i + bs]
            c, l, q = ctx_all[sl].to(DEVICE), clen_all[sl].to(DEVICE), cp_all[sl].to(DEVICE); rb = real[i:i + bs].to(DEVICE); g = []
            for t in range(H):
                top = masked_logits(model, c, l).topk(10, 1).indices; a = top[:, 0]
                per["hit"][t].append((top == rb[:, t:t + 1]).any(1).float().cpu())
                per["rmA"][t].append(rm_candidates(rm, c, l, a[:, None], q)[:, 0].cpu())
                per["rmB"][t].append(rm_candidates(rm_b, c, l, a[:, None], q)[:, 0].cpu())
                per["lpop"][t].append(lpop_t[a].cpu()); g.append(a)
                c, q, l = append_tok(c, l, q, a if mode == "free" else rb[:, t])
            gen.append(torch.stack(g, 1).cpu())
        gen = torch.cat(gen).numpy(); cat = lambda L: torch.cat(L).numpy()
        res[mode] = {k: [cat(per[k][t]) for t in range(H)] for k in per}
        res[mode]["seq_recall"] = np.array([len(set(g) & set(r)) / H for g, r in zip(gen, real.numpy())])
        res[mode]["uniq_share"] = [len(np.unique(gen[:, t])) / len(gen) for t in range(H)]
    return res

ms_names = ["base"] + [n for n in ("pct_b0.5", "pct+hit", "long_b0.1", "pct_H5") if n in rl_results]
def load_policy(n):
    if n == "base": return base
    m = new_model(); m.load_state_dict(torch.load(CKPT/f"rl_{n}_{RUN_TAG}.pt", map_location=DEVICE)); return m.eval()
ms = {}
for n in ms_names:
    ms[n] = multistep(load_policy(n))
print("пользователей в многошаговой оценке:", ms["base"]["n_users"])

ms_rows = []
for n, r in ms.items():
    for mode in ("free", "teacher"):
        for t in range(H_MS):
            ms_rows.append(dict(model=n, mode=mode, step=t + 1, recall10=boot_ci(r[mode]["hit"][t]), rmA=boot_ci(r[mode]["rmA"][t]),
                                rmB=float(r[mode]["rmB"][t].mean()), uniq_share=r[mode]["uniq_share"][t], mean_log_pop=float(r[mode]["lpop"][t].mean()),
                                seq_recall=float(r[mode]["seq_recall"].mean())))
json.dump(ms_rows, open(RESULTS/f"multistep_{RUN_TAG}.json", "w"), indent=1)
ms_df = pd.DataFrame(ms_rows)
ms_df["Recall@10"] = ms_df.recall10.map(lambda v: f"{v[0]:.4f} [{v[1]:.4f}, {v[2]:.4f}]")
ms_df["RM-A топ-1"] = ms_df.rmA.map(lambda v: f"{v[0]:.3f}")
ms_df[ms_df["mode"] == "free"].pivot(index="model", columns="step", values="Recall@10")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
cols_ms = dict(zip(ms_names, ["#222222", "#0072B2", "#009E73", "#D55E00", "#56B4E9"]))
for n in ms_names:
    for mode, ls in (("free", "-"), ("teacher", "--")):
        d = ms_df[(ms_df.model == n) & (ms_df["mode"] == mode)]
        ax[0].plot(d.step, d.recall10.map(lambda v: v[0]), ls, color=cols_ms[n], marker="o", ms=3, label=f"{n} ({mode})")
        ax[1].plot(d.step, d.rmA.map(lambda v: v[0]), ls, color=cols_ms[n], marker="o", ms=3)
        if mode == "free": ax[2].plot(d.step, d.uniq_share, ls, color=cols_ms[n], marker="o", ms=3)
ax[0].set(title="Recall@10 по шагам (сплошная: free, пунктир: teacher)", xlabel="шаг", ylabel="Recall@10")
ax[1].set(title="Reward RM-A топ-1 по шагам", xlabel="шаг", ylabel="RM-A")
ax[2].set(title="Доля уникальных объектов среди пользователей (free)", xlabel="шаг", ylabel="уникальных / пользователей")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(RESULTS/f"fig_multistep_{RUN_TAG}.png", dpi=130); plt.show()
pd.DataFrame({n: {"seq_recall free": ms[n]["free"]["seq_recall"].mean(), "seq_recall teacher": ms[n]["teacher"]["seq_recall"].mean()} for n in ms}).T.round(4)

### Выводы по разделу 6

_Заполнить после полного прогона на Kaggle._

## 7. Варианты функции вознаграждения и компромисс

**7.1** Компромисс: для каждого прогона прирост reward (по RM-A) против изменения Recall@10 с ДИ. Порог «заметной потери» — снижение Recall@10 больше чем на 0.005 (≈2.4% от базового значения).

**7.2** Финальная оценка на **test** (открывается один раз): базовая модель, бейзлайны и все RL-прогоны, одна и та же функция оценки и один и тот же протокол. Ничего не подбиралось по test.

In [ ]:
THR = 0.005
tr_ = pd.DataFrame({n: {"ΔRM-A": d["ΔRM-A"][0], "ΔRecall@10": d["ΔRecall@10"][0], "ΔNDCG@10": d["ΔNDCG@10"][0]} for n, d in delta.items()}).T
tr_["потеря заметна"] = np.where(tr_["ΔRecall@10"] < -THR, "да", "нет")
tr_["прирост reward на 0.01 потери Recall"] = np.where(tr_["ΔRecall@10"] < 0, tr_["ΔRM-A"] / (-tr_["ΔRecall@10"] * 100), np.nan)
display(tr_.sort_values("ΔRM-A", ascending=False).round(4))

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.axhspan(-THR, 0.02, color="#009E73", alpha=.08)           # зона допустимой потери релевантности
for n, d in delta.items():
    (x, xl, xh), (y, yl, yh) = d["ΔRM-A"], d["ΔRecall@10"]; col, mk = fam(n)
    ax.errorbar(x, y, xerr=[[x - xl], [xh - x]], yerr=[[y - yl], [yh - y]], fmt=mk, color=col, capsize=2, ms=7)
    ax.annotate(n, (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.axhline(0, color="#888", lw=.8); ax.axvline(0, color="#888", lw=.8); ax.axhline(-THR, color="#009E73", lw=.8, ls="--")
ax.set(xlabel="Δ reward RM-A (топ-10)", ylabel="Δ Recall@10", title="Прирост reward против потери релевантности (val)")
plt.tight_layout(); plt.savefig(RESULTS/f"fig_tradeoff_{RUN_TAG}.png", dpi=130); plt.show()

### 7.2 Финальная оценка на test

In [ ]:
ranks_b, recs_b = evaluate(gpt_scores(base), "test")
test_res = {"base": eval_recs("base", ranks_b, recs_b, "test")}; BASE_RECS_TEST = recs_b
pop_all_t = torch.tensor(item_cnt, dtype=torch.float32, device=DEVICE)
pop_7d_t = torch.tensor(np.bincount(df[df.period == 1].item.values, minlength=n_items) + 1e-6 * item_cnt, dtype=torch.float32, device=DEVICE)  # последняя неделя до test
for nm, vec in (("Pop (all time)", pop_all_t), ("Pop (last 7d)", pop_7d_t)):
    r_, rec_ = evaluate(lambda c, l, v=vec: v[None].expand(len(c), -1), "test")
    test_res[nm] = eval_recs(nm.replace(" ", "_"), r_, rec_, "test", BASE_RECS_TEST)
for n in runs:
    pol = load_policy(n); test_res[n], _ = full_eval(n, pol, "test", base_recs=BASE_RECS_TEST); del pol
    if DEVICE == "cuda": torch.cuda.empty_cache()
json.dump(test_res, open(RESULTS/f"test_{RUN_TAG}.json", "w"), indent=1)
rl_table(test_res)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for n, r in test_res.items():
    x = r["reward@10"]["RM-A"][0]; y, lo, hi = r["rank_metrics"]["Recall@10"]
    col, mk = ("#222222", "*") if n == "base" else (("#888888", "x") if n.startswith("Pop") else fam(n))
    ax.errorbar(x, y, yerr=[[y - lo], [hi - y]], fmt=mk, color=col, capsize=2, ms=9 if n == "base" else 7)
    ax.annotate(n, (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set(xlabel="средний reward RM-A в топ-10", ylabel="Recall@10", title="Test: reward против релевантности")
plt.tight_layout(); plt.savefig(RESULTS/f"fig_test_{RUN_TAG}.png", dpi=130); plt.show()

### Выводы по разделу 7

_Заполнить после полного прогона на Kaggle._

## 8. Выводы и ограничения